# posthoc_weights: weight-space edits between the student and its base

Training-free edits that use the student's weights and the raw base it was distilled from (`modernbert-tr-base-1k`): `theta = base + gamma * (student - base)` per tensor (`turkish_distill.weightedit`). gamma = 1 is the student, below 1 pulls toward the base (WiSE-FT style interpolation), above 1 extrapolates past the student along the path training took. Reading the weights first showed that distillation moved every layer by 14-26% of its norm, most in the MLPs of the last layers (23-26%).

Four profiles are written to Drive `models/<name>` (no GPU needed; the original `final/` folders are not touched):

| name | gamma | tensors |
|---|---|---|
| `wise_all_g90` | 0.90 | everything, embeddings and final norm included |
| `extrap_all_g110` | 1.10 | everything |
| `wise_late_g75` | 0.75 | layers 16-21, all modules |
| `wise_mlp_late_g75` | 0.75 | layers 16-21, MLP only |

**Predictions, written before any score exists.** (1) Interpolating toward the raw base (48.8 on TR-MTEB) lowers Mean(task), most on retrieval and STS, which the late layers carry; a gain would mean distillation overshot. (2) Extrapolating is the mirror image and a cheap stand-in for more training steps: the budget runs gained +0.39 from 5400 to 10,800 steps, so a small positive effect (below ~0.4) is plausible; if interpolation and extrapolation both lose, the student sits at a local optimum along this line and the idea is dead. (3) Anything within 0.3 of 67.71 is noise-level and is not interpreted.

**Round structure (the model proposes, you evaluate).** Round 1 = `wise_all_g90` and `extrap_all_g110` (a symmetric +-10% test of the direction). Paste the TR-MTEB lines back; round 2 (the late-layer profiles, or a different gamma) is chosen from what they show. The selection happens on TR-MTEB, so anything that works must be confirmed on Mizan's independent tasks before it is reported.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'      # local disk, never Drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib, os, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell first.')
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')       # an editable install's .pth is not read by a running session
importlib.invalidate_caches()
os.environ.setdefault('HF_HOME', '/content/hf_cache')
import turkish_distill.weightedit               # fails here if the repo on GitHub does not have the module yet

In [ ]:
# --- read both weight sets ---
import re, shutil
import torch
from huggingface_hub import snapshot_download
from safetensors import safe_open
from safetensors.torch import save_file
from turkish_distill.weightedit import PROFILES, scale_delta, summarise, relative_change

MODEL = f'{DRIVE_ROOT}/runs/scale_two_instr8k_s10800_lr1e4/20261027-152516-b2b0c4b/final'   # main line (TR-MTEB 67.71)
BASE_REPO, BASE_REV = 'ytu-ce-cosmos/modernbert-tr-base-1k', 'c2e35f553101cce5b03fbb318ef8728473fd34dd'
OUT_MODELS = f'{DRIVE_ROOT}/models'
assert os.path.isdir(MODEL), MODEL

def load(path, strip_prefix=False):
    out = {}
    with safe_open(path, 'pt', device='cpu') as f:
        for k in f.keys():
            out[re.sub(r'^model\.', '', k) if strip_prefix else k] = f.get_tensor(k)
    return out

student = load(f'{MODEL}/model.safetensors')
base_dir = snapshot_download(BASE_REPO, revision=BASE_REV, allow_patterns=['*.safetensors', 'config.json'], local_dir='/content/models/base1k')
base = load(f'{base_dir}/model.safetensors', strip_prefix=True)
print(len(student), 'student tensors;', len(set(student) - set(base)), 'missing in the base')
assert not set(student) - set(base)

In [ ]:
# --- build the profiles; a gamma-1 control must reproduce the student exactly ---
control = scale_delta(student, base, dict(gamma=1.0, layers=None, groups=None, outside_layers=True))
assert all(torch.equal(control[k], student[k]) for k in student), 'gamma = 1 changed something'
print('control (gamma = 1) reproduces the student exactly')

KEEP = [f for f in os.listdir(MODEL) if f not in ('model.safetensors', 'teacher_heads.pt')]    # config and tokenizer files
for name, spec in PROFILES.items():
    out = f'{OUT_MODELS}/{name}'
    if os.path.isdir(out):
        print(f'{name:<20} exists, skipped')
        continue
    edited = scale_delta(student, base, spec)
    os.makedirs(out)
    for f in KEEP:
        shutil.copy(f'{MODEL}/{f}', f'{out}/{f}')
    save_file({k: v.contiguous() for k, v in edited.items()}, f'{out}/model.safetensors', metadata={'format': 'pt'})
    print(f'{name:<20} {summarise(edited, student, spec)}  -> {out}')

In [ ]:
# --- load every written folder the way the evaluation does, compare its vectors with the student's on 64 training texts ---
import json, random
import numpy as np
from sentence_transformers import SentenceTransformer

MIX = f'{DRIVE_ROOT}/data/mix/mix_v2_200k.jsonl'
texts = [json.loads(line)['text'] for line in open(MIX, encoding='utf-8')][:64]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def vectors(path):
    m = SentenceTransformer(path, device=device)
    m.max_seq_length = 512
    return m.encode(texts, batch_size=32, convert_to_numpy=True)

ref = vectors(MODEL)
print(f'{"model":<20}{"modules":<26}{"cosine to the student (mean / min)":>36}')
for name in PROFILES:
    v = vectors(f'{OUT_MODELS}/{name}')
    cos = (v * ref).sum(1) / (np.linalg.norm(v, axis=1) * np.linalg.norm(ref, axis=1))
    mods = [type(m).__name__ for m in SentenceTransformer(f'{OUT_MODELS}/{name}', device='cpu')]
    print(f'{name:<20}{str(mods):<26}{cos.mean():>20.4f} / {cos.min():.4f}')
print('\nEval presets (already in eval3.ipynb; add them to another eval notebook if you use one):')
for name in PROFILES:
    print(f"    '{name}': dict(model=f'{{DRIVE_ROOT}}/models/{name}'),")

In [ ]:
from google.colab import runtime
runtime.unassign()